# LAB 08 — Tokenization bằng BERT

Notebook này quan sát trực tiếp cách tokenizer biến câu thành subword tokens, token IDs và attention mask. Chạy các cell theo thứ tự.

## 1. Nạp tokenizer và xác định thư mục dữ liệu

In [1]:
from pathlib import Path
import sys
from transformers import AutoTokenizer

PROJECT_DIR = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'data' / 'IMDB Dataset.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Mở notebook trong dự án NLP để tìm thấy data/.')
LAB_DIR = PROJECT_DIR / 'lab08'
MODEL_NAME = 'google/bert_uncased_L-2_H-128_A-2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer:', MODEL_NAME)
print('Vocabulary size:', tokenizer.vocab_size)
print('Special tokens:', tokenizer.special_tokens_map)

C:\Users\KIEN\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer: google/bert_uncased_L-2_H-128_A-2
Vocabulary size: 30522
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## 2. Tokenize câu trong đề: “The cats are playing.”

In [2]:
sentence = 'The cats are playing.'
words = sentence.split()
subword_tokens = tokenizer.tokenize(sentence)
encoded = tokenizer(sentence, return_tensors='pt')
input_ids = encoded['input_ids'][0].tolist()
attention_mask = encoded['attention_mask'][0].tolist()
all_tokens = tokenizer.convert_ids_to_tokens(input_ids)

print('Câu:', sentence)
print('Số từ tách theo khoảng trắng:', len(words))
print('Subword tokens:', subword_tokens)
print('Số subword tokens:', len(subword_tokens))
print('Tokens có special tokens:', all_tokens)
print('Token IDs:', input_ids)
print('Special tokens:', tokenizer.all_special_tokens)
print('Số special tokens được thêm:', len(all_tokens) - len(subword_tokens))
print('Attention mask:', attention_mask)

Câu: The cats are playing.
Số từ tách theo khoảng trắng: 4
Subword tokens: ['the', 'cats', 'are', 'playing', '.']
Số subword tokens: 5
Tokens có special tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Token IDs: [101, 1996, 8870, 2024, 2652, 1012, 102]
Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
Số special tokens được thêm: 2
Attention mask: [1, 1, 1, 1, 1, 1, 1]


## 3. Kiểm tra input cho một batch

In [3]:
sentences = [
    'The cats are playing.',
    'This movie was surprisingly good.',
]
batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors='pt',
)
print('Các key:', list(batch.keys()))
print('Shape input_ids:', tuple(batch['input_ids'].shape))
print('Shape attention_mask:', tuple(batch['attention_mask'].shape))
for i, sentence in enumerate(sentences):
    print(f'\nCâu {i + 1}: {sentence}')
    print('Tokens:', tokenizer.convert_ids_to_tokens(batch['input_ids'][i]))
    print('IDs:', batch['input_ids'][i].tolist())
    print('Mask:', batch['attention_mask'][i].tolist())

Các key: ['input_ids', 'token_type_ids', 'attention_mask']
Shape input_ids: (2, 8)
Shape attention_mask: (2, 8)

Câu 1: The cats are playing.
Tokens: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]']
IDs: [101, 1996, 8870, 2024, 2652, 1012, 102, 0]
Mask: [1, 1, 1, 1, 1, 1, 1, 0]

Câu 2: This movie was surprisingly good.
Tokens: ['[CLS]', 'this', 'movie', 'was', 'surprisingly', 'good', '.', '[SEP]']
IDs: [101, 2023, 3185, 2001, 10889, 2204, 1012, 102]
Mask: [1, 1, 1, 1, 1, 1, 1, 1]
